In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import numpy.ma as ma
import pandas as pd
import scipy as sp
import seaborn as sns
from scipy import stats
from scipy.stats import bootstrap
#import bayes_toolbox.glm as bg
# import arviz as az
import statsmodels.api as sm
from statsmodels.formula.api import ols

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

In [ ]:
def set_default():
    sns.set_theme(context="paper", font_scale=1.2)
    sns.set_style("ticks")
    plt.rcParams["mathtext.default"] = "regular"
    plt.rc("axes.spines", top=False, right=False)

In [ ]:
# Outlier removal function

def remove_outliers(df, var, z_thresh):

    # z-score
    df[var + "_z"] = df.groupby("SN")[var].transform(stats.zscore)

    # Create outlier column
    df[var + "_outlier"] = (
        (np.abs(df[var + "_z"]) > z_thresh) 
    )

    # Calculate within-subject mean using non-outlier trials only
    df[var + "_mean"] = (
        df[(np.abs(df[var + "_z"]) <= z_thresh)]
        .groupby("SN")[var].transform("mean")
    )

    # Replace outliers with NaNs
    df[var + "_clean"] = np.where(
        (np.abs(df[var + "_z"]) > z_thresh),
        np.nan,
        df[var])

    # Create col that replaces outliers with within-subject mean values (may not use)
    df[var + "_mean"] = df.groupby("SN")[var + "_mean"].transform(lambda x: x.fillna(np.nanmean(x)))

    # Count number of outliers per participant
    num_outliers = df.groupby("SN")[var + "_outlier"].sum()
    print(num_outliers)

    # Print proportion of trials removed
    print(num_outliers / df["TN"].max())

In [ ]:
def motor_sd(df, numBaseline, cleanData):
    mask = (df["TN"] > (numBaseline - 50)) & (df["TN"] <= numBaseline)
    df["motor_sd"] = df[mask].groupby("SN")[cleanData].transform("std")
    df[mask]
    df.head()

In [ ]:
def binning(df, var, clean):
    df["quintile"] = df.groupby(["SN", var], observed=False)[clean].transform(
        lambda x: pd.qcut(x, 5, labels=range(1,6)))
    
    df_binned = df.groupby(["SN", var, "quintile"], observed=False)[[clean, "adaptation"]].mean().reset_index()

In [ ]:
def regress(df, var):

    beta_binned_var = np.zeros(16)

    for idx, val in enumerate(df["SN"].unique()):
        subj_var = df.loc[df["SN"] == val, :]
        model_var = ols("adaptation ~ " + var + "- 1", subj_var).fit()
        beta_binned_var[idx] = model_var._results.params[0]

    beta_binned_var = (beta_binned_var,)
    res_var = bootstrap(beta_binned_var, np.mean, method="percentile").confidence_interval

    print(f"95\% CI for " + var + f": {res_var}")

In [ ]:
def popn(df, var, quintile):
    df.groupby(["var", quintile], as_index=False, observed=False).mean().drop(columns="SN")

In [ ]:
def plot_coeffs(ax, data, y1, y2):
    sns.pointplot(data=data, x="EGE", y=y1, ax=ax, c="r")
    sns.stripplot(data=data, x="EGE", y=y1, alpha=0.3, ax=ax, c="r")
    sns.pointplot(data=data, x="IGE", y=y2, ax=ax, c="b")
    sns.stripplot(data=data, x="IGE", y=y2, alpha=0.3, ax=ax, c="b")
    ax.axhline(linewidth=0.5, color="k")
    plt.tight_layout()
    return ax

def set_labels(fig, ax, title=None):
    ax.set(xlabel="", ylabel=r"$\beta$ (sensitivity)", title=title, ylim=(-0.25, 0.85))
    sns.despine()
    
def set_style():
    # This sets reasonable defaults for font size for
    # a figure that will go in a paper
    sns.set_theme(context="paper", font_scale=1.2)
    sns.set_style("ticks")


def plot_binned_data(binned_data, popn_data, ege_col, beta_ege_col, beta_ige_col):
    '''
    Group-level figure showing relationships between adaptation and IGE/EGE.
    '''

    # Set-up color cycles
    colors_ige = plt.get_cmap("Blues")(np.linspace(0.2, 0.8, 5))
    colors_ege = plt.get_cmap("Reds")(np.linspace(0.2, 0.8, 5))
    
    # Plot adaptive response vs IGE and EGE for rotation trials
    perts = np.unique(binned_data[ege_col])
    pert = []
    x_mean_ige = np.zeros(5)
    x_err_ige = np.zeros(5)
    y_mean_ige = np.zeros(5)
    y_err_ige = np.zeros(5)
    x_mean_ege = np.zeros(5)
    x_err_ege = np.zeros(5)
    y_mean_ege = np.zeros(5)
    y_err_ege = np.zeros(5)
    h = np.zeros(len(perts), dtype=int)
    
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(7, 2.75), width_ratios=[3, 3, 1])
    for i in range(len(perts)):
        idx = binned_data[ege_col] == perts[i]
        pert = pd.DataFrame(binned_data.loc[idx, :])
        pert["ige_mean"] = np.nan
        for j in range(5):
            idx_bin = pert["ige_quintile"] == j + 1
            temp = pert.loc[idx_bin, :]
            x_mean_ige[j] = pert.loc[idx_bin, "theta_maxradv_clean"].mean()
            x_err_ige[j] = pert.loc[idx_bin, "theta_maxradv_clean"].sem()
            y_mean_ige[j] = pert.loc[idx_bin, "adaptation"].mean()
            y_err_ige[j] = pert.loc[idx_bin, "adaptation"].sem()
            pert.loc[idx_bin, "ige_mean"] = pert.loc[idx_bin, "theta_maxradv_clean"].mean()
            ax2.errorbar(x=x_mean_ige[j], y=y_mean_ige[j], yerr=y_err_ige[j], ecolor=colors_ege[i],
                     **{"marker":"o", "markeredgecolor":colors_ege[i], "markerfacecolor":colors_ige[j], "linestyle":"none"}) 
            ax2.set_xticks((-4, -2, 0, 2, 4))
        sns.regplot(data=pert, x="ige_mean", y="adaptation", ax=ax2, ci=0, 
                    scatter=False, scatter_kws={"color":colors_ege[i]}, 
                    line_kws={"color":colors_ege[i], "linewidth":1})
    # Plot overall correlation and compute statistics, bootstrapped CIs
    sns.regplot(data=popn_data, x="theta_maxradv_clean", y="adaptation", ax=ax2,
                scatter=False, ci=None, line_kws={"color":"k", "linestyle":"--"})
    slope_ige, _, r_ige, p_ige, std_err_ige = stats.linregress(
        popn_data["theta_maxradv_clean"], popn_data["adaptation"])
    ax2.text(-4, 3.5, f"$r^2$={r_ige**2:.3f}", fontsize="small")
    ax2.text(-4, 3.0, f"$slope=${slope_ige:.3f}", fontsize="small")
    
    # Outer loop is for ige bins, inner loop for pert levels
    for i in range(5):
        idx = binned_data["ige_quintile"] == i + 1
        binned = pd.DataFrame(binned_data.loc[idx, :])
        for j in range(5):
            idx_ege = binned[ege_col] == perts[j]
            x_mean_ege[j] = binned.loc[idx_ege, ege_col].mean()
            x_err_ege[j] = binned.loc[idx_ege, ege_col].sem()
            y_mean_ege[j] = binned.loc[idx_ege, "adaptation"].mean()
            y_err_ege[j] = binned.loc[idx_ege, "adaptation"].sem()
            binned.loc[idx_ege, "ege_mean"] = binned.loc[idx_ege, ege_col].mean()
            ax1.errorbar(x=x_mean_ege[j], y=y_mean_ege[j], yerr=y_err_ege[j], ecolor=colors_ege[i],
                    **{"marker":"o", "markeredgecolor":colors_ege[j], "markerfacecolor":colors_ige[i], "linestyle":"none"})
            ax1.set_xticks((-4, -2, 0, 2, 4))
        sns.regplot(data=binned, x="ege_mean", y="adaptation", ax=ax1, ci=0,
                    scatter=False, line_kws={"color":colors_ige[i], "linewidth":1})
    sns.regplot(data=popn_data, x=ege_col, y="adaptation", ax=ax1,
                scatter=False, ci=None, line_kws={"color":"k", "linestyle":"--"})
    # Plot overall correlation and compute statistics, bootstrapped CIs
    r_ege, p_ege = sp.stats.pearsonr(popn_data[ege_col], popn_rotation["adaptation"])
    slope_ege, _, r_ege, p_ege, std_err_ege = stats.linregress(
        popn_data[ege_col], popn_data["adaptation"])
    ax1.text(-4, -1.5, f"$r^2$={r_ege**2:.3f}", fontsize="small")
    ax1.text(-4, -2.0, f"$slope=${slope_ege:.3f}", fontsize="small")
    
    # More figure aesthetics    
    ax1.set(xlabel="Externally-generated error ($\degree$)", ylabel="Adaptation ($\degree$)", xlim=[-4.5, 4.5], ylim=[-4, 4])
    sns.despine()
    ax2.set(xlabel="Internally-generated error ($\degree$)", ylabel="Adaptation ($\degree$)", xlim=[-4.5, 4.5], ylim=[-4, 4])
    plt.tight_layout()
    
    ax3 = plot_coeffs(ax3, df_betas, df_betas[beta_ege_col], df_betas[beta_ige_col])
    set_labels(fig, ax3)
    ax3.tick_params(axis="x", labelrotation=45)
    plt.tight_layout()

    return fig